# Diagnostics, Controls, and the Honest Baseline
### Experiment v4a — acquisition-cue audit and factorial ablation

This version introduces no new mechanism. It tests whether v3's result is attributable
to the representation or to confounds in the pipeline and the dataset.

v3 reported raw pixels at balanced accuracy 0.9072 and AUC 0.9950, and HOG at 0.7686,
reversing the 6buwan ordering in which HOG performed better. Three alternative
explanations are examined before that result can be interpreted.

| # | Alternative explanation | Status at the outset | Settled by |
|---|---|---|---|
| 1 | v3 omits the global feature standardisation present in v2, so dissimilarity measures are dominated by the highest-variance dimensions | **Confirmed in code**: `build_features` returns unstandardised features and `pair_descriptor` consumes them directly | Part 4 and Part 8 |
| 2 | The eight band descriptors are spatially coherent for raw pixels, whose vector is a row-major image, but spatially incoherent for HOG, whose `11×11×2×2×9` layout is flattened, so half the descriptor treats the two representations differently | **Confirmed in code**: `np.array_split` operates on the flat vector | Part 5 and Part 8 |
| 3 | The raw-pixel false acceptance rate of 0.0000 derives from ink quantity and stroke-weight cues that HOG's L2-Hys block normalisation discards, making it an acquisition artifact rather than shape discrimination | Unverified | Part 7 |

Two further changes concern validity rather than performance.

**A development and confirmation split across writers.** v1, v2 and v3 were each
developed by inspecting test results on all 55 writers, which constitutes
researcher-degrees-of-freedom leakage at the study level and is not visible in any
individual run's protocol. From this version onward, 25 writers are development and 30
are reserved. The confirmation writers are evaluated once, at the end of the study.

**No selection on test performance.** v3's target sweep showed `t = 0.90` producing the
highest test balanced accuracy. Adopting it on that basis would constitute selection on
the evaluation set. The reported target remains `t = 0.70`, specified before any result
was observed, and the sweep is reported as a sensitivity analysis.

## Scope

This version contains no calibration experiments, no cohort-margin selection, no
learned descriptor weighting and no attribution. Its purpose is to establish which
representation is free of the identified confounds, and how much of the apparent
performance is obtainable from trivial image statistics. A high AUC from the
trivial-feature control in Part 7 would indicate that the headline result requires
reinterpretation.

## Environment

Scikit-learn's `OneClassSVM` is a CPU-only estimator with no GPU or TPU backend, so an
accelerator runtime confers no benefit and may be slower. A CPU, High-RAM runtime is
used. Feature extraction is cached to Drive as `.npy`, removing the 28.5-minute cold
read that dominated v3's runtime.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import numpy as np, pandas as pd
from PIL import Image, ImageOps
from math import floor, ceil, sqrt
from skimage.feature import hog
from skimage.filters import threshold_otsu
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import random, time, itertools, platform, sys
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)

NB_T0 = time.time(); SECTION_TIMES = {}
def mark(section):
    now = time.time(); prev = SECTION_TIMES.get("_last", NB_T0)
    SECTION_TIMES[section] = now - prev; SECTION_TIMES["_last"] = now
    print(f"[timer] {section}: {now-prev:.1f}s   (total {now-NB_T0:.1f}s)", flush=True)

print("v4 imports loaded. Seed =", RANDOM_SEED)
print("Python", sys.version.split()[0], "|", platform.processor() or "cpu")
print("scikit-learn is CPU-only -- OneClassSVM has no GPU/TPU path. Use CPU + High-RAM.")

v4 imports loaded. Seed = 42
Python 3.13.16 | x86_64
scikit-learn is CPU-only -- OneClassSVM has no GPU/TPU path. Use CPU + High-RAM.


In [ ]:
DATASET_DIR = Path("/content/drive/MyDrive/cedar-dataset/signatures")
GENUINE_DIR = DATASET_DIR / "full_org"
FORGED_DIR  = DATASET_DIR / "full_forg"

CACHE_DIR = Path("/content/drive/MyDrive/4buwan_cache"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR   = Path("/content/drive/MyDrive/4buwan_results_v4"); OUT_DIR.mkdir(parents=True, exist_ok=True)
USE_CACHE = True          # set False to force re-extraction

IMG_SIZE        = (100, 100)
MAX_CROP_ASPECT = 2.5
DENSITY_FRAC    = 0.05
N_WRITERS       = 55
N_TRAIN_GENUINE = 12

# ---- unchanged from v3 so the replication arms are exact ----
PAIR_BASELINE = {"nu": 0.10, "gamma": "scale"}

# ---- v4: development / confirmation writer split ----
SPLIT_SEED  = 20261014        # fixed the day the split was made; never change this
N_DEV       = 25

# ---- descriptor config ----
DESC_GRID   = (3, 3)          # spatial regions for the corrected block descriptors
INK_THRESH  = 0.2

print("Config set.")
print(f"Cache : {CACHE_DIR}")
print(f"Output: {OUT_DIR}")

Config set.
Cache : /content/drive/MyDrive/4buwan_cache
Output: /content/drive/MyDrive/4buwan_results_v4


## Part 1 — Data, per-writer splits, and the locked confirmation set

The per-writer split is byte-identical to v3: 12 enrolled genuine, 12 held-out
genuine, 24 forged. Nothing about an individual writer changes.

What is new is the split **across** writers. The 55 writers are partitioned once,
under a fixed seed, into 25 development and 30 confirmation writers. Every
diagnostic, control and design decision from here on is made on development
writers only. The confirmation writers exist to be run once at the end of v5/v6.

This is the only protection against the one leak that v3's clean per-writer
protocol does not cover: the fact that v1, v2 and v3 were themselves selected by
looking at test results on all 55 writers.

In [4]:
genuine_files = sorted(GENUINE_DIR.glob("*.png"))
forged_files  = sorted(FORGED_DIR.glob("*.png"))
def get_writer_id(fp): return int(Path(fp).stem.split("_")[1])
assert len(genuine_files) == 1320 and len(forged_files) == 1320
print("Files OK:", len(genuine_files), "genuine /", len(forged_files), "forged")

rng_split = random.Random(RANDOM_SEED); writer_splits = {}
for wid in range(1, N_WRITERS+1):
    g = sorted([p for p in genuine_files if get_writer_id(p)==wid])
    f = sorted([p for p in forged_files  if get_writer_id(p)==wid])
    sg = g.copy(); rng_split.shuffle(sg); sf = f.copy(); rng_split.shuffle(sf)
    writer_splits[wid] = {"train_genuine": sg[:N_TRAIN_GENUINE],
                          "test_genuine":  sg[N_TRAIN_GENUINE:],
                          "test_forged":   sf}
for s in writer_splits.values():
    assert len(s["train_genuine"])==12 and len(s["test_genuine"])==12 and len(s["test_forged"])==24
print("Per-writer split OK -- identical to v3 (12 / 12 / 24)")

# ---------------- development / confirmation writer split ----------------
_perm   = np.random.default_rng(SPLIT_SEED).permutation(np.arange(1, N_WRITERS+1))
DEV     = sorted(int(w) for w in _perm[:N_DEV])
CONFIRM = sorted(int(w) for w in _perm[N_DEV:])
assert set(DEV).isdisjoint(CONFIRM) and len(DEV)+len(CONFIRM) == N_WRITERS

print(f"\nDEV     ({len(DEV)} writers): {DEV}")
print(f"CONFIRM ({len(CONFIRM)} writers): {CONFIRM}")
print("\n" + "!"*72)
print("CONFIRM writers are LOCKED. No diagnostic, control, descriptor choice or")
print("hyperparameter in v4 or v5 may be selected by looking at their results.")
print("They are evaluated exactly once, at the end, and that run is Chapter 4.")
print("!"*72)

pd.DataFrame({"partition": ["DEV"]*len(DEV) + ["CONFIRM"]*len(CONFIRM),
              "writer": DEV + CONFIRM}).to_csv(OUT_DIR/"writer_partition.csv", index=False)
mark("Part 1 splits")

Files OK: 1320 genuine / 1320 forged
Per-writer split OK -- identical to v3 (12 / 12 / 24)

DEV     (25 writers): [3, 8, 9, 10, 12, 17, 19, 21, 22, 23, 28, 29, 34, 36, 39, 40, 42, 43, 44, 47, 49, 50, 51, 52, 55]
CONFIRM (30 writers): [1, 2, 4, 5, 6, 7, 11, 13, 14, 15, 16, 18, 20, 24, 25, 26, 27, 30, 31, 32, 33, 35, 37, 38, 41, 45, 46, 48, 53, 54]

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CONFIRM writers are LOCKED. No diagnostic, control, descriptor choice or
hyperparameter in v4 or v5 may be selected by looking at their results.
They are evaluated exactly once, at the end, and that run is Chapter 4.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
[timer] Part 1 splits: 7.1s   (total 7.1s)


## Part 2 — Preprocessing and feature extraction (cached)

Preprocessing is unchanged from v3: invert, Otsu-threshold to find ink, crop to a
density-filtered bounding box, pad to an aspect ratio of at most 2.5, resize and
centre on a 100x100 canvas.

Features are extracted for **all 55 writers** and cached to Drive as `.npy`.
v3 spent 1707.7s here against 36.5s in v2 for identical work — that gap was cold
Google Drive I/O, not compute. With the cache in place, reruns load in seconds,
and v5/v6 reuse the same arrays.

Geometry statistics for the Part 7 ink control are extracted in the same pass, so
the control sees exactly the same preprocessed images as the real pipeline.

In [5]:
def _common_geometry(img, max_aspect=MAX_CROP_ASPECT, density_frac=DENSITY_FRAC, img_size=IMG_SIZE):
    tw, th = img_size
    ink = img > threshold_otsu(img) if img.max() > img.min() else np.zeros_like(img, bool)
    if ink.any():
        rc, cc = ink.sum(1), ink.sum(0)
        sr = rc >= max(2, rc.max()*density_frac); sc = cc >= max(2, cc.max()*density_frac)
        if sr.any() and sc.any():
            t,b = np.where(sr)[0][[0,-1]]; l,r = np.where(sc)[0][[0,-1]]
            img = img[t:b+1, l:r+1]
    h,w = img.shape
    if w/h > max_aspect:
        pad = max(0, ceil(w/max_aspect)-h); img = np.pad(img, ((pad//2, pad-pad//2),(0,0)), constant_values=0)
    elif h/w > max_aspect:
        pad = max(0, ceil(h/max_aspect)-w); img = np.pad(img, ((0,0),(pad//2, pad-pad//2)), constant_values=0)
    h,w = img.shape
    nw = min(tw, max(1, w*th//h)); nh = max(1, h*nw//w)
    img = np.asarray(Image.fromarray(img.astype(np.uint8)).resize((nw,nh)))
    top=floor((th-nh)/2); bot=ceil((th-nh)/2); lef=floor((tw-nw)/2); rig=ceil((tw-nw)/2)
    return np.pad(img, ((top,bot),(lef,rig)), mode="constant", constant_values=0)

def preprocess_signature(path):
    with Image.open(path) as im:
        return _common_geometry(np.asarray(ImageOps.invert(im.convert("L"))))

HOG_PARAMS = dict(orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), block_norm="L2-Hys")
def raw_feature_fn(i01): return i01.flatten()
def hog_feature_fn(i01): return hog(i01, **HOG_PARAMS)

GEOM_NAMES = ["ink_total","ink_area","cent_y","cent_x","spread_y","spread_x",
              "bbox_aspect","mean_ink_intensity","fill_density"]

def geom_stats(i01, thr=INK_THRESH):
    """Deliberately trivial image statistics -- the ink/geometry negative control."""
    ink = i01 > thr
    tot  = float(i01.sum()); area = float(ink.sum())
    ys, xs = np.nonzero(ink)
    if len(ys) == 0: return np.zeros(len(GEOM_NAMES), dtype=np.float32)
    h = float(ys.max()-ys.min()+1); w = float(xs.max()-xs.min()+1)
    return np.array([tot, area, float(ys.mean()), float(xs.mean()),
                     float(ys.std()), float(xs.std()), w/max(h,1.0),
                     tot/max(area,1.0), tot/max(h*w,1.0)], dtype=np.float32)

def _extract(paths):
    """One preprocessing pass producing RAW, HOG and geometry features together."""
    R, H, G = [], [], []
    for k, p in enumerate(paths):
        i01 = preprocess_signature(p).astype(np.float32)/255.0
        R.append(raw_feature_fn(i01)); H.append(hog_feature_fn(i01)); G.append(geom_stats(i01))
        if (k+1) % 400 == 0: print(f"    ...{k+1}/{len(paths)}", flush=True)
    return (np.asarray(R, dtype=np.float32), np.asarray(H, dtype=np.float32),
            np.asarray(G, dtype=np.float32))

SPLIT_KEYS = [("train", "train_genuine"), ("test_gen", "test_genuine"), ("test_for", "test_forged")]
FILES = {k: [p for w in range(1, N_WRITERS+1) for p in writer_splits[w][src]] for k, src in SPLIT_KEYS}

F, GEO = {"RAW": {}, "HOG": {}}, {}
for key in FILES:
    names = {r: CACHE_DIR/f"{r}_{key}.npy" for r in ("RAW","HOG","GEO")}
    if USE_CACHE and all(f.exists() for f in names.values()):
        F["RAW"][key] = np.load(names["RAW"]); F["HOG"][key] = np.load(names["HOG"])
        GEO[key] = np.load(names["GEO"])
        print(f"  {key}: loaded from cache  RAW {F['RAW'][key].shape}  HOG {F['HOG'][key].shape}")
    else:
        print(f"  {key}: extracting {len(FILES[key])} images...", flush=True)
        r, h, g = _extract(FILES[key])
        np.save(names["RAW"], r); np.save(names["HOG"], h); np.save(names["GEO"], g)
        F["RAW"][key], F["HOG"][key], GEO[key] = r, h, g
        print(f"  {key}: extracted + cached  RAW {r.shape}  HOG {h.shape}")

HOG_NB = int(round(sqrt(F["HOG"]["train"].shape[1] / (HOG_PARAMS["cells_per_block"][0] *
              HOG_PARAMS["cells_per_block"][1] * HOG_PARAMS["orientations"]))))
print(f"\nHOG block grid inferred: {HOG_NB} x {HOG_NB} blocks x "
      f"{HOG_PARAMS['cells_per_block'][0]*HOG_PARAMS['cells_per_block'][1]*HOG_PARAMS['orientations']} "
      f"= {HOG_NB*HOG_NB*36}")
assert HOG_NB*HOG_NB*36 == F["HOG"]["train"].shape[1], "HOG reshape would be invalid"
mark("Part 2 features")

  train: extracting 660 images...
    ...400/660
  train: extracted + cached  RAW (660, 10000)  HOG (660, 4356)
  test_gen: extracting 660 images...
    ...400/660
  test_gen: extracted + cached  RAW (660, 10000)  HOG (660, 4356)
  test_for: extracting 1320 images...
    ...400/1320
    ...800/1320
    ...1200/1320
  test_for: extracted + cached  RAW (1320, 10000)  HOG (1320, 4356)

HOG block grid inferred: 11 x 11 blocks x 36 = 4356
[timer] Part 2 features: 1253.3s   (total 1260.5s)


## Part 3 — Global feature standardisation (the confirmed v3 regression)

v2 z-scored every feature dimension using a scaler fitted on pooled training
genuine signatures, and HOG reached AUC 0.95 in signature space. v3 computes
cosine, Pearson, normalised Euclidean, mean-absolute-difference and chi-square
**directly on raw features**, so each of those measures is dominated by whichever
dimensions happen to carry the largest raw variance. Informative dimensions get no
extra weight.

The per-writer `StandardScaler` that v3 applies inside `fit_pairs_params` cannot
repair this: it acts on the already-computed 16-D descriptor matrix, long after
the variance domination has been baked into the descriptors.

The scaler here is fitted on **development writers' enrolled genuine signatures
only** — never on test signatures, never on confirmation writers. It is then
frozen and reused unchanged in v5 and v6, so the confirmation run inherits a
scaler that never saw it.

In [6]:
SCALERS, FS = {}, {"RAW": {}, "HOG": {}}

def _row_index(key, writers):
    """Row positions in F[rep][key] belonging to the given writers."""
    wset = set(writers)
    return np.array([i for i, p in enumerate(FILES[key]) if get_writer_id(p) in wset], dtype=int)

dev_train_rows = _row_index("train", DEV)
print(f"Scaler fit pool: {len(dev_train_rows)} signatures "
      f"({len(DEV)} DEV writers x {N_TRAIN_GENUINE} enrolled genuine)")

for rep in ("RAW", "HOG"):
    sc = StandardScaler().fit(F[rep]["train"][dev_train_rows].astype(np.float64))
    SCALERS[rep] = sc
    for key in F[rep]:
        FS[rep][key] = sc.transform(F[rep][key].astype(np.float64)).astype(np.float32)
    nz = int((sc.scale_ > 1e-8).sum())
    print(f"  {rep}: {F[rep]['train'].shape[1]} dims, {nz} with non-degenerate variance "
          f"({100*nz/F[rep]['train'].shape[1]:.1f}%)")

print("\nSparsity (relevant to the HOG hypotheses):")
for rep in ("RAW","HOG"):
    X = F[rep]["train"][dev_train_rows]
    print(f"  {rep}: fraction of exactly-zero entries = {float((X==0).mean()):.3f}   "
          f"mean value = {float(X.mean()):.4f}")
mark("Part 3 standardisation")

Scaler fit pool: 300 signatures (25 DEV writers x 12 enrolled genuine)
  RAW: 10000 dims, 10000 with non-degenerate variance (100.0%)
  HOG: 4356 dims, 4356 with non-degenerate variance (100.0%)

Sparsity (relevant to the HOG hypotheses):
  RAW: fraction of exactly-zero entries = 0.525   mean value = 0.0465
  HOG: fraction of exactly-zero entries = 0.447   mean value = 0.0717
[timer] Part 3 standardisation: 0.7s   (total 1261.1s)


## Part 4 — Pair descriptors: legacy (v3) and spatially corrected

Both descriptor families share the same 8 global measures. They differ only in how
the locality-aware block terms are formed.

**Legacy (exactly v3).** `np.array_split(vec, 8)` on the flat feature vector.
For RAW the vector is a row-major 100x100 image, so the 8 chunks are 8 clean
horizontal bands — genuinely spatial. For HOG the vector is `(11, 11, 2, 2, 9)`
flattened, so 8 chunks of 544.5 straddle block-row boundaries and mix orientation
bins together. Half of the 16-D descriptor is therefore spatially coherent for RAW
and spatially smeared for HOG. That is a property of the array layout, not of the
features, and it is a second mechanism by which v3 could have manufactured RAW's
advantage.

**Spatial (corrected).** Each feature vector is reshaped to its true spatial grid
— `(100, 100)` for RAW, `(11, 11, 36)` for HOG — and split into a 3x3 region grid.
Regional Pearson correlations are then computed between corresponding regions, so
both representations receive the same spatial treatment.

Keeping both families lets Part 8 attribute the RAW/HOG gap to standardisation, to
the block layout, or to neither, as a clean 2x2x2 factorial rather than a guess.

In [7]:
EPS = 1e-12

def _globals8(a, b):
    out = []
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    out.append(float(a @ b / (na*nb + EPS)))                         # cosine
    am, bm = a - a.mean(), b - b.mean()
    out.append(float(am @ bm / (np.linalg.norm(am)*np.linalg.norm(bm) + EPS)))   # Pearson
    rho = spearmanr(a, b).correlation                                # Starovoitov-style
    out.append(float(rho) if np.isfinite(rho) else 0.0)
    out.append(float(np.linalg.norm(a-b) / (na + nb + EPS)))         # normalised Euclidean
    d = np.abs(a-b)
    out.append(float(d.mean())); out.append(float(d.std()))          # MAD, MAD dispersion
    ap, bp = np.clip(a, 0, None), np.clip(b, 0, None)
    ap = ap/(ap.sum()+EPS); bp = bp/(bp.sum()+EPS)
    out.append(float(np.minimum(ap, bp).sum()))                      # histogram intersection
    out.append(float(0.5 * ((ap-bp)**2 / (ap+bp+EPS)).sum()))        # chi-square
    return out

GLOBAL_NAMES = ["cos","pearson","spearman","eucn","mad","mad_sd","intersect","chi2"]

def _corr(x, y):
    x = x - x.mean(); y = y - y.mean()
    return float(x @ y / (np.linalg.norm(x)*np.linalg.norm(y) + EPS))

# ---------------- legacy: exactly v3 ----------------
N_BLOCKS_LEGACY = 8
def pair_descriptor_legacy(a, b, rep=None):
    a = np.asarray(a, np.float64); b = np.asarray(b, np.float64)
    out = _globals8(a, b)
    for ab, bb in zip(np.array_split(a, N_BLOCKS_LEGACY), np.array_split(b, N_BLOCKS_LEGACY)):
        out.append(_corr(ab, bb))
    return np.array(out)

# ---------------- spatial: corrected ----------------
def _as_grid(vec, rep):
    if rep == "RAW": return vec.reshape(IMG_SIZE[1], IMG_SIZE[0])
    return vec.reshape(HOG_NB, HOG_NB, -1)

def _regions(A, gy, gx):
    for ys in np.array_split(np.arange(A.shape[0]), gy):
        for xs in np.array_split(np.arange(A.shape[1]), gx):
            yield A[ys][:, xs].reshape(-1)

def pair_descriptor_spatial(a, b, rep):
    a = np.asarray(a, np.float64); b = np.asarray(b, np.float64)
    out = _globals8(a, b)
    gy, gx = DESC_GRID
    for ra, rb in zip(_regions(_as_grid(a, rep), gy, gx), _regions(_as_grid(b, rep), gy, gx)):
        out.append(_corr(ra, rb))
    return np.array(out)

DESCRIPTORS = {"legacy": pair_descriptor_legacy, "spatial": pair_descriptor_spatial}
DESC_DIMS   = {"legacy": 8 + N_BLOCKS_LEGACY, "spatial": 8 + DESC_GRID[0]*DESC_GRID[1]}
DESC_NAMES  = {"legacy":  GLOBAL_NAMES + [f"band{i}" for i in range(N_BLOCKS_LEGACY)],
               "spatial": GLOBAL_NAMES + [f"reg{r}{c}" for r in range(DESC_GRID[0])
                                                        for c in range(DESC_GRID[1])]}

for k in DESCRIPTORS:
    d = len(DESCRIPTORS[k](np.random.rand(10000), np.random.rand(10000), "RAW"))
    assert d == DESC_DIMS[k] == len(DESC_NAMES[k])
    print(f"  {k:8s}: {d}-D  ({DESC_NAMES[k]})")

print(f"\nPer writer: C(12,2) = 66 enrolled pairs")
print(f"  samples-per-dimension: legacy {66/DESC_DIMS['legacy']:.1f}, "
      f"spatial {66/DESC_DIMS['spatial']:.1f}   (v2 signature space: {12/4356:.4f})")

  legacy  : 16-D  (['cos', 'pearson', 'spearman', 'eucn', 'mad', 'mad_sd', 'intersect', 'chi2', 'band0', 'band1', 'band2', 'band3', 'band4', 'band5', 'band6', 'band7'])
  spatial : 17-D  (['cos', 'pearson', 'spearman', 'eucn', 'mad', 'mad_sd', 'intersect', 'chi2', 'reg00', 'reg01', 'reg02', 'reg10', 'reg11', 'reg12', 'reg20', 'reg21', 'reg22'])

Per writer: C(12,2) = 66 enrolled pairs
  samples-per-dimension: legacy 4.1, spatial 3.9   (v2 signature space: 0.0028)


## Part 5 — Building the eight arms

A full 2 (representation) x 2 (standardisation) x 2 (descriptor layout) factorial,
on development writers only. Every arm's descriptors are computed once and cached
in memory, because Parts 6 and 8 both read them: the per-descriptor diagnostic and
the fixed-baseline evaluation must be computed from identical descriptor matrices
or they cannot be compared.

Per writer per arm: 66 enrolled pairs, 12x12 genuine query pairs, 24x12 forged
query pairs — 498 descriptor evaluations. The Spearman term over a 10,000-element
vector is the expensive part, which is why nothing is recomputed downstream.

In [8]:
ARMS = [(rep, sc, de) for rep in ("RAW","HOG") for sc in ("unscaled","scaled")
                        for de in ("legacy","spatial")]
def arm_name(a): return f"{a[0]}-{a[1]}-{a[2]}"

def _feat(rep, scaling): return F[rep] if scaling == "unscaled" else FS[rep]

def build_arm(rep, scaling, desc, writers=DEV, verbose=True):
    """Per writer: D (66,dim), Qg (12,12,dim), Qf (24,12,dim)."""
    fn = DESCRIPTORS[desc]
    Xtr, Xtg, Xtf = (_feat(rep, scaling)[k] for k in ("train","test_gen","test_for"))
    store = {}
    for n, wid in enumerate(writers, 1):
        W  = Xtr[_row_index("train", [wid])].astype(np.float64)
        Tg = Xtg[_row_index("test_gen", [wid])].astype(np.float64)
        Tf = Xtf[_row_index("test_for", [wid])].astype(np.float64)
        idx = list(itertools.combinations(range(len(W)), 2))
        D  = np.array([fn(W[i], W[j], rep) for i, j in idx])
        Qg = np.array([[fn(q, t, rep) for t in W] for q in Tg])
        Qf = np.array([[fn(q, t, rep) for t in W] for q in Tf])
        store[wid] = {"D": D, "idx": idx, "Qg": Qg, "Qf": Qf}
        if verbose and n % 5 == 0: print(f"    ...{n}/{len(writers)} writers", flush=True)
    return store

DESCSETS = {}
for a in ARMS:
    t0 = time.time()
    print(f"Building arm {arm_name(a)} ...", flush=True)
    DESCSETS[a] = build_arm(*a)
    print(f"  done in {time.time()-t0:.1f}s", flush=True)
mark("Part 5 descriptor sets")

Building arm RAW-unscaled-legacy ...
    ...5/25 writers
    ...10/25 writers
    ...15/25 writers
    ...20/25 writers
    ...25/25 writers
  done in 37.3s
Building arm RAW-unscaled-spatial ...
    ...5/25 writers
    ...10/25 writers
    ...15/25 writers
    ...20/25 writers
    ...25/25 writers
  done in 39.2s
Building arm RAW-scaled-legacy ...
    ...5/25 writers
    ...10/25 writers
    ...15/25 writers
    ...20/25 writers
    ...25/25 writers
  done in 52.5s
Building arm RAW-scaled-spatial ...
    ...5/25 writers
    ...10/25 writers
    ...15/25 writers
    ...20/25 writers
    ...25/25 writers
  done in 57.5s
Building arm HOG-unscaled-legacy ...
    ...5/25 writers
    ...10/25 writers
    ...15/25 writers
    ...20/25 writers
    ...25/25 writers
  done in 27.1s
Building arm HOG-unscaled-spatial ...
    ...5/25 writers
    ...10/25 writers
    ...15/25 writers
    ...20/25 writers
    ...25/25 writers
  done in 31.4s
Building arm HOG-scaled-legacy ...
    ...5/25 writers
    

## Part 6 — Diagnostic: what is each descriptor actually worth?

Each of the 16 or 17 descriptor coordinates is scored on its own, as a single
number separating a writer's held-out genuine signatures from their forgeries.
Direction is irrelevant — a coordinate that is perfectly *inverted* is just as
informative — so each AUC is reported sign-free as `max(auc, 1-auc)`, and 0.5
means the coordinate carries nothing.

How to read the table:

- **If `mad`, `mad_sd` and `eucn` carry RAW while `cos` and `pearson` are weak**,
  RAW is running on magnitude, i.e. ink quantity and stroke weight. That is the
  ink hypothesis, and Part 7 then decides how much of the headline it explains.
- **If `spearman` sits near 0.5 for HOG**, the tied-zeros problem is confirmed:
  rank correlation over a vector that is mostly exactly zero measures the layout
  of empty blocks rather than stroke structure.
- **If standardisation lifts the HOG column substantially**, hypothesis 1 is the
  dominant cause and the v3 ordering was an artifact of a missing scaler.

The descriptor correlation matrix is printed alongside. The 66 enrolled pairs come
from only 12 signatures, so each signature appears in 11 pairs and the rows are
strongly dependent — the effective sample size is far closer to 12 than to 66.
Highly correlated coordinates in that regime add noise to the per-writer scaler
rather than information, which is the case for pruning them in v5.

In [9]:
def per_descriptor_auc(arm, writers=DEV):
    store, dim = DESCSETS[arm], None
    rows = []
    for wid in writers:
        s = store[wid]
        g = s["Qg"].mean(axis=1)           # (12, dim) -- mean over the 12 enrolled pairs
        f = s["Qf"].mean(axis=1)           # (24, dim)
        dim = g.shape[1]
        y = np.r_[np.ones(len(g)), np.zeros(len(f))]
        per = []
        for k in range(dim):
            s_k = np.r_[g[:, k], f[:, k]]
            if np.allclose(s_k, s_k[0]): per.append(0.5); continue
            a = roc_auc_score(y, s_k); per.append(max(a, 1.0-a))
        rows.append(per)
    return np.mean(rows, axis=0)

desc_auc = {}
for a in ARMS:
    desc_auc[a] = pd.Series(per_descriptor_auc(a), index=DESC_NAMES[a[2]])

for de in ("legacy","spatial"):
    cols = {f"{rep}-{sc}": desc_auc[(rep,sc,de)] for rep in ("RAW","HOG") for sc in ("unscaled","scaled")}
    tbl = pd.DataFrame(cols)
    print("="*78); print(f"PER-DESCRIPTOR AUC  --  {de} block layout   (sign-free, 0.5 = useless)")
    print("="*78)
    print(tbl.round(3).to_string())
    print(f"\n  best single coordinate per arm:")
    for c in tbl.columns:
        print(f"    {c:16s} {tbl[c].idxmax():10s} {tbl[c].max():.3f}      "
              f"(mean over coords {tbl[c].mean():.3f})")
    print()
    tbl.to_csv(OUT_DIR/f"per_descriptor_auc_{de}.csv")

print("="*78); print("DESCRIPTOR REDUNDANCY (spatial, scaled) -- |corr| over pooled enrolled pairs")
print("="*78)
for rep in ("RAW","HOG"):
    Dall = np.vstack([DESCSETS[(rep,"scaled","spatial")][w]["D"] for w in DEV])
    Cm = np.abs(np.corrcoef(Dall, rowvar=False))
    iu = np.triu_indices_from(Cm, k=1)
    print(f"  {rep}: mean |corr| off-diagonal = {np.nanmean(Cm[iu]):.3f}   "
          f"pairs with |corr|>0.9 = {int(np.nansum(Cm[iu] > 0.9))}/{len(iu[0])}")
    pd.DataFrame(Cm, index=DESC_NAMES["spatial"], columns=DESC_NAMES["spatial"]).round(3) \
      .to_csv(OUT_DIR/f"descriptor_corr_{rep}.csv")
mark("Part 6 diagnostic")

PER-DESCRIPTOR AUC  --  legacy block layout   (sign-free, 0.5 = useless)
           RAW-unscaled  RAW-scaled  HOG-unscaled  HOG-scaled
cos               1.000       0.957         0.946       0.956
pearson           0.992       0.985         0.936       0.951
spearman          0.922       0.985         0.919       0.951
eucn              1.000       0.958         0.949       0.954
mad               0.985       0.963         0.928       0.920
mad_sd            0.814       0.793         0.935       0.939
intersect         1.000       0.989         0.959       0.956
chi2              1.000       0.989         0.960       0.957
band0             0.571       0.579         0.631       0.645
band1             0.622       0.665         0.692       0.811
band2             0.799       0.854         0.903       0.929
band3             0.895       0.912         0.925       0.941
band4             0.863       0.862         0.916       0.923
band5             0.754       0.834         0.899       0.9

## Part 7 — The ink / geometry negative control

This is the control that decides whether the v3 headline survives.

RAW's false acceptance rate at the fixed baseline was **0.0000** — not one false
accept in 1,320 forged queries, before any calibration. Five of the sixteen
descriptor coordinates (normalised Euclidean, mean absolute difference and its
dispersion, histogram intersection, chi-square) are magnitude-sensitive, and on
raw pixels magnitude means ink. If CEDAR's forgeries differ systematically from
genuine signatures in total ink, stroke weight or extent, those coordinates can
separate them without representing signature *shape* at all — and HOG, whose
L2-Hys normalisation rescales every block to unit length, has deliberately thrown
exactly those cues away. That alone could explain the entire RAW/HOG reversal.

So: replace the 10,000-dimensional image with nine deliberately trivial scalars —
total ink, inked area, centroid, spread, bounding-box aspect, mean ink intensity,
fill density — and push them through the **identical** pair / OCSVM / majority-vote
pipeline. No shape information of any kind survives this representation.

| Control AUC | Reading |
|---|---|
| ~0.60–0.75 | The pair representation is doing real work. The v3 result stands. |
| ~0.85–0.95 | Most of the headline is low-level acquisition cues. Move the headline to HOG and report this as the finding. |
| >0.95 | CEDAR's skilled forgeries are trivially separable. The entire performance literature on this dataset needs the caveat, and that becomes a contribution in its own right. |

Context worth having while reading it: published CEDAR results now include 99.77%
accuracy (Starovoitov & Akhundjanov 2024, using per-writer one-class SVM on pair
rank-correlations) and 100% (Bhavani et al. 2026). Two independent studies
reaching near-perfect forgery rejection with pairwise similarity descriptors is
itself a reason to suspect the dataset rather than the methods. No one in this
line has published the control.

In [10]:
def geom_pair_descriptor(a, b, rep=None):
    """Symmetric, scale-free comparison of two trivial geometry vectors."""
    out = []
    for i in range(len(a)):
        x, y = float(a[i]), float(b[i])
        s = abs(x) + abs(y) + EPS
        out.append(abs(x-y)/s)
        out.append(min(abs(x),abs(y))/(max(abs(x),abs(y)) + EPS))
    return np.array(out)

GEOM_PAIR_NAMES = [f"{n}_{k}" for n in GEOM_NAMES for k in ("absdiff","ratio")]

def build_geom_arm(writers=DEV):
    store = {}
    for wid in writers:
        W  = GEO["train"][_row_index("train", [wid])].astype(np.float64)
        Tg = GEO["test_gen"][_row_index("test_gen", [wid])].astype(np.float64)
        Tf = GEO["test_for"][_row_index("test_for", [wid])].astype(np.float64)
        idx = list(itertools.combinations(range(len(W)), 2))
        store[wid] = {"D":  np.array([geom_pair_descriptor(W[i], W[j]) for i, j in idx]),
                      "idx": idx,
                      "Qg": np.array([[geom_pair_descriptor(q, t) for t in W] for q in Tg]),
                      "Qf": np.array([[geom_pair_descriptor(q, t) for t in W] for q in Tf])}
    return store

INK_ARM = ("INK","unscaled","geom")
DESCSETS[INK_ARM] = build_geom_arm()
DESC_NAMES["geom"] = GEOM_PAIR_NAMES
DESC_DIMS["geom"]  = len(GEOM_PAIR_NAMES)

ink_auc = pd.Series(per_descriptor_auc(INK_ARM), index=GEOM_PAIR_NAMES)
print("="*78); print("INK / GEOMETRY CONTROL -- per-coordinate AUC (sign-free)")
print("="*78)
print(ink_auc.round(3).sort_values(ascending=False).to_string())
print(f"\n  strongest trivial cue: {ink_auc.idxmax()} at AUC {ink_auc.max():.3f}")
ink_auc.to_csv(OUT_DIR/"ink_control_per_descriptor_auc.csv")
mark("Part 7 ink control descriptors")

INK / GEOMETRY CONTROL -- per-coordinate AUC (sign-free)
fill_density_absdiff          0.999
fill_density_ratio            0.999
ink_total_ratio               0.986
ink_total_absdiff             0.986
mean_ink_intensity_ratio      0.953
mean_ink_intensity_absdiff    0.953
ink_area_absdiff              0.792
ink_area_ratio                0.790
spread_y_ratio                0.758
spread_y_absdiff              0.757
bbox_aspect_absdiff           0.757
bbox_aspect_ratio             0.756
cent_y_absdiff                0.740
cent_y_ratio                  0.739
spread_x_absdiff              0.717
spread_x_ratio                0.717
cent_x_absdiff                0.646
cent_x_ratio                  0.645

  strongest trivial cue: fill_density_absdiff at AUC 0.999
[timer] Part 7 ink control descriptors: 1.2s   (total 1580.0s)


## Part 8 — Fixed-baseline evaluation, all nine arms

Identical protocol to v3 so the numbers are directly comparable: one OCSVM per
writer fitted on that writer's 66 enrolled pairs with a per-writer scaler,
`nu = 0.10` and `gamma = "scale"` — both chosen before any result was seen and held
constant across every arm. A questioned signature is paired against each of the 12
enrolled signatures and decided by **majority vote**; the threshold-free score for
AUC is the mean decision function over those 12 pairs. AUC is computed per writer
and then averaged, which is correct for writer-dependent systems because pooling
raw scores across writers would conflate per-writer score scales.

No calibration anywhere in this part. The point is to see what each representation
is worth before any adaptation, with the two confirmed bugs switched on and off.

In [11]:
def _decide(preds): return 1 if int((preds == 1).sum()) > len(preds)/2 else -1

def evaluate_arm(arm, writers=DEV, nu=None, gamma=None):
    nu    = PAIR_BASELINE["nu"]    if nu    is None else nu
    gamma = PAIR_BASELINE["gamma"] if gamma is None else gamma
    store, rows = DESCSETS[arm], []
    for wid in writers:
        s  = store[wid]
        sc = StandardScaler().fit(s["D"])
        m  = OneClassSVM(kernel="rbf", nu=nu, gamma=gamma).fit(sc.transform(s["D"]))
        def run(Q):
            dec, sco = [], []
            for q in Q:
                Qs = sc.transform(q)
                dec.append(_decide(m.predict(Qs))); sco.append(float(m.decision_function(Qs).mean()))
            return np.array(dec), np.array(sco)
        gd, gs = run(s["Qg"]); fd, fs = run(s["Qf"])
        tp = int((gd==1).sum()); fn = int((gd==-1).sum())
        fp = int((fd==1).sum()); tn = int((fd==-1).sum())
        frr = fn/(tp+fn); far = fp/(fp+tn)
        prec = tp/(tp+fp) if (tp+fp) else 0.0; rec = tp/(tp+fn) if (tp+fn) else 0.0
        rows.append({"Writer": wid, "FRR": frr, "FAR": far,
                     "Accuracy": (tp+tn)/(tp+tn+fp+fn),
                     "F1": 2*prec*rec/(prec+rec) if (prec+rec) else 0.0,
                     "AUC": roc_auc_score(np.r_[np.ones(len(gs)), np.zeros(len(fs))], np.r_[gs, fs])})
    df = pd.DataFrame(rows)
    ov = {k: float(df[k].mean()) for k in ("FRR","FAR","Accuracy","F1","AUC")}
    ov["BalancedAccuracy"] = ((1-ov["FRR"]) + (1-ov["FAR"]))/2
    return df, ov

ALL_ARMS = ARMS + [INK_ARM]
per_writer, overall = {}, []
for a in ALL_ARMS:
    t0 = time.time()
    df, ov = evaluate_arm(a)
    per_writer[a] = df
    overall.append({"Arm": arm_name(a), "Rep": a[0], "Scaling": a[1], "Descriptor": a[2],
                    "dim": DESC_DIMS[a[2]], **ov})
    print(f"  {arm_name(a):22s} BalAcc {ov['BalancedAccuracy']:.4f}  AUC {ov['AUC']:.4f}  "
          f"FRR {ov['FRR']:.4f}  FAR {ov['FAR']:.4f}   [{time.time()-t0:.0f}s]", flush=True)

summary = pd.DataFrame(overall)
print("\n" + "="*96)
print(f"FIXED-BASELINE RESULTS  --  {len(DEV)} DEV writers, nu={PAIR_BASELINE['nu']}, gamma={PAIR_BASELINE['gamma']}")
print("="*96)
print(summary.round(4).to_string(index=False))
summary.to_csv(OUT_DIR/"v4_arm_summary.csv", index=False)
for a in ALL_ARMS: per_writer[a].to_csv(OUT_DIR/f"per_writer_{arm_name(a)}.csv", index=False)
mark("Part 8 arm evaluation")

  RAW-unscaled-legacy    BalAcc 0.8050  AUC 0.9974  FRR 0.3900  FAR 0.0000   [1s]
  RAW-unscaled-spatial   BalAcc 0.7850  AUC 0.9976  FRR 0.4300  FAR 0.0000   [1s]
  RAW-scaled-legacy      BalAcc 0.8108  AUC 0.9783  FRR 0.3733  FAR 0.0050   [1s]
  RAW-scaled-spatial     BalAcc 0.8158  AUC 0.9861  FRR 0.3667  FAR 0.0017   [1s]
  HOG-unscaled-legacy    BalAcc 0.7700  AUC 0.8553  FRR 0.3133  FAR 0.1467   [1s]
  HOG-unscaled-spatial   BalAcc 0.7383  AUC 0.8540  FRR 0.3867  FAR 0.1367   [1s]
  HOG-scaled-legacy      BalAcc 0.8033  AUC 0.8871  FRR 0.2700  FAR 0.1233   [1s]
  HOG-scaled-spatial     BalAcc 0.7300  AUC 0.8642  FRR 0.4067  FAR 0.1333   [1s]
  INK-unscaled-geom      BalAcc 0.7117  AUC 1.0000  FRR 0.5767  FAR 0.0000   [1s]

FIXED-BASELINE RESULTS  --  25 DEV writers, nu=0.1, gamma=scale
                 Arm Rep  Scaling Descriptor  dim    FRR    FAR  Accuracy     F1    AUC  BalancedAccuracy
 RAW-unscaled-legacy RAW unscaled     legacy   16 0.3900 0.0000    0.8700 0.7447 0.9974    

## Part 9 — Attribution of the v3 RAW-over-HOG gap, and the verdict

The factorial lets the gap be decomposed instead of guessed at. Three quantities
matter:

1. **Standardisation effect**, within each representation and descriptor layout:
   how much does fitting the global scaler before computing descriptors change AUC?
   A large positive effect on HOG confirms hypothesis 1.
2. **Block-layout effect**: legacy minus spatial. A RAW advantage that shrinks when
   both representations get the same spatial treatment confirms hypothesis 2.
3. **Residual RAW-over-HOG gap** in the fully corrected arm. Whatever survives both
   fixes is the gap that needs a substantive explanation — and the ink control
   says whether that explanation is "RAW sees stroke weight".

In [12]:
def get(rep, sc, de, metric="AUC"):
    return float(summary[(summary.Rep==rep)&(summary.Scaling==sc)&(summary.Descriptor==de)][metric].iloc[0])

print("="*78); print("1. STANDARDISATION EFFECT (scaled - unscaled), AUC")
print("="*78)
for rep in ("RAW","HOG"):
    for de in ("legacy","spatial"):
        d = get(rep,"scaled",de) - get(rep,"unscaled",de)
        print(f"  {rep} / {de:8s}: {get(rep,'unscaled',de):.4f} -> {get(rep,'scaled',de):.4f}   ({d:+.4f})")

print("\n" + "="*78); print("2. BLOCK-LAYOUT EFFECT (spatial - legacy), AUC")
print("="*78)
for rep in ("RAW","HOG"):
    for sc in ("unscaled","scaled"):
        d = get(rep,sc,"spatial") - get(rep,sc,"legacy")
        print(f"  {rep} / {sc:8s}: {get(rep,sc,'legacy'):.4f} -> {get(rep,sc,'spatial'):.4f}   ({d:+.4f})")

print("\n" + "="*78); print("3. RAW-over-HOG GAP, by arm")
print("="*78)
gaps = {}
for sc in ("unscaled","scaled"):
    for de in ("legacy","spatial"):
        g = get("RAW",sc,de) - get("HOG",sc,de); gaps[(sc,de)] = g
        tag = "  <-- v3's configuration" if (sc,de)==("unscaled","legacy") else \
              ("  <-- both fixes applied" if (sc,de)==("scaled","spatial") else "")
        print(f"  {sc:8s} / {de:8s}: RAW {get('RAW',sc,de):.4f}  HOG {get('HOG',sc,de):.4f}   gap {g:+.4f}{tag}")

v3_gap, fixed_gap = gaps[("unscaled","legacy")], gaps[("scaled","spatial")]
explained = (v3_gap - fixed_gap)/v3_gap*100 if abs(v3_gap) > 1e-9 else float("nan")
print(f"\n  gap closed by the two code fixes: {explained:.0f}% "
      f"({v3_gap:+.4f} -> {fixed_gap:+.4f})")

ink = summary[summary.Rep=="INK"].iloc[0]
best = summary[summary.Rep!="INK"].sort_values("AUC", ascending=False).iloc[0]
print("\n" + "="*78); print("4. INK / GEOMETRY CONTROL")
print("="*78)
print(f"  nine trivial scalars    : AUC {ink.AUC:.4f}   BalAcc {ink.BalancedAccuracy:.4f}   FAR {ink.FAR:.4f}")
print(f"  best real representation: AUC {best.AUC:.4f}   BalAcc {best.BalancedAccuracy:.4f}   ({best.Arm})")
share = (ink.AUC-0.5)/max(best.AUC-0.5, 1e-9)*100
print(f"  trivial cues account for {share:.0f}% of the best arm's AUC above chance")

print("\n" + "="*78); print("VERDICT")
print("="*78)
if ink.AUC > 0.95:
    print("  [CRITICAL] CEDAR forgeries are near-trivially separable on ink/geometry alone.")
    print("             Every performance number on this dataset -- ours, Starovoitov's 99.77%,")
    print("             Bhavani's 100% -- needs this caveat. Report the control as a FINDING,")
    print("             move the headline to the representation that does NOT ride on it, and")
    print("             add a matched-FAR or ink-normalised evaluation in v5.")
elif ink.AUC > 0.85:
    print("  [SERIOUS]  Most of the apparent performance is low-level acquisition cues.")
    print("             Do not headline RAW. Report the control, headline the shape-based arm,")
    print("             and frame the control as a methodological contribution.")
elif ink.AUC > 0.75:
    print("  [CAUTION]  Trivial cues carry real signal but not the result. Report the control")
    print("             in Chapter 4 as a validity check and keep the headline, with the caveat.")
else:
    print("  [CLEAR]    Trivial cues do not explain the result. The pair representation is")
    print("             doing genuine work and the v3 headline survives this challenge.")
print()
if abs(fixed_gap) < 0.02:
    print("  RAW and HOG are equivalent once both code bugs are fixed. The v3 'reversal' was")
    print("  an artifact of a missing scaler and an unfair block layout, NOT a finding.")
    print("  6buwan's ordering is not contradicted. Do not write the reversal paragraph.")
elif fixed_gap > 0:
    print(f"  RAW still leads by {fixed_gap:+.4f} AUC with both fixes applied. The reversal is")
    print("  real and needs the mechanism explanation -- read it together with the ink control")
    print("  above, which says whether that mechanism is stroke weight.")
else:
    print(f"  HOG leads by {-fixed_gap:+.4f} AUC once both fixes are applied. 6buwan's ordering")
    print("  is RESTORED and HOG is the honest headline. v5 is built on HOG.")
print("\n  Chosen arm for v5:", best.Arm)
mark("Part 9 verdict")

1. STANDARDISATION EFFECT (scaled - unscaled), AUC
  RAW / legacy  : 0.9974 -> 0.9783   (-0.0190)
  RAW / spatial : 0.9976 -> 0.9861   (-0.0115)
  HOG / legacy  : 0.8553 -> 0.8871   (+0.0318)
  HOG / spatial : 0.8540 -> 0.8642   (+0.0101)

2. BLOCK-LAYOUT EFFECT (spatial - legacy), AUC
  RAW / unscaled: 0.9974 -> 0.9976   (+0.0003)
  RAW / scaled  : 0.9783 -> 0.9861   (+0.0078)
  HOG / unscaled: 0.8553 -> 0.8540   (-0.0012)
  HOG / scaled  : 0.8871 -> 0.8642   (-0.0229)

3. RAW-over-HOG GAP, by arm
  unscaled / legacy  : RAW 0.9974  HOG 0.8553   gap +0.1421  <-- v3's configuration
  unscaled / spatial : RAW 0.9976  HOG 0.8540   gap +0.1436
  scaled   / legacy  : RAW 0.9783  HOG 0.8871   gap +0.0912
  scaled   / spatial : RAW 0.9861  HOG 0.8642   gap +0.1219  <-- both fixes applied

  gap closed by the two code fixes: 14% (+0.1421 -> +0.1219)

4. INK / GEOMETRY CONTROL
  nine trivial scalars    : AUC 1.0000   BalAcc 0.7117   FAR 0.0000
  best real representation: AUC 0.9976   BalAcc 0.7

## Part 10 — Runtime and environment report

In [13]:
total = time.time() - NB_T0; h, rem = divmod(total, 3600); m, s = divmod(rem, 60)
print("="*70); print("RUNTIME REPORT -- 4buwan v4"); print("="*70)
print(f"  Environment : Google Colaboratory, High-RAM, Python {sys.version.split()[0]}")
print(f"  Accelerator : CPU (scikit-learn OneClassSVM has no GPU/TPU backend --")
print(f"                selecting GPU or TPU gives no speedup here and can be slower)")
print(f"  Writers     : {len(DEV)} development  /  {len(CONFIRM)} confirmation (LOCKED)")
print(f"  Arms        : {len(ALL_ARMS)} ({len(ARMS)} factorial + 1 ink control)")
print(f"  Feature cache: {'HIT -- loaded from Drive' if USE_CACHE else 'bypassed'}")
print("-"*70)
for k, v in SECTION_TIMES.items():
    if k == "_last": continue
    print(f"  {k:34s} {v:8.1f}s   ({v/60:5.2f} min)")
print("-"*70)
print(f"  TOTAL: {int(h)}h {int(m)}m {int(s)}s   ({total/60:.1f} minutes)")
print("="*70)

pd.DataFrame([{"section": k, "seconds": v} for k, v in SECTION_TIMES.items() if k != "_last"]) \
  .to_csv(OUT_DIR/"v4_runtime.csv", index=False)
print(f"\nAll artifacts written to {OUT_DIR}")
for f in sorted(OUT_DIR.glob("*.csv")): print("  ", f.name)

RUNTIME REPORT -- 4buwan v4
  Environment : Google Colaboratory, High-RAM, Python 3.13.16
  Accelerator : CPU (scikit-learn OneClassSVM has no GPU/TPU backend --
                selecting GPU or TPU gives no speedup here and can be slower)
  Writers     : 25 development  /  30 confirmation (LOCKED)
  Arms        : 9 (8 factorial + 1 ink control)
  Feature cache: HIT -- loaded from Drive
----------------------------------------------------------------------
  Part 1 splits                           7.1s   ( 0.12 min)
  Part 2 features                      1253.3s   (20.89 min)
  Part 3 standardisation                  0.7s   ( 0.01 min)
  Part 5 descriptor sets                312.5s   ( 5.21 min)
  Part 6 diagnostic                       5.1s   ( 0.09 min)
  Part 7 ink control descriptors          1.2s   ( 0.02 min)
  Part 8 arm evaluation                   6.4s   ( 0.11 min)
  Part 9 verdict                          0.1s   ( 0.00 min)
---------------------------------------------------

---
## Reported quantities

The four verdict blocks in Part 9, together with the two per-descriptor AUC tables in
Part 6, settle three questions.

1. **Which representation is free of the identified confounds**, and therefore which
   one subsequent experiments are built on.
2. **Whether the reversal between signature space and dissimilarity space is a finding
   or an artifact.** If the gap between representations closes once both confirmed code
   faults are corrected, the reversal is not a result and 6buwan's ordering stands.
3. **How much of the separation on this dataset is obtainable from trivial image
   statistics.** The published state of the art on CEDAR is 99.77% (Starovoitov and
   Akhundjanov, 2024) and 100% (Bhavani, Bharathi and Manjunatha, 2026), neither
   reporting such a control.

Nothing in this version constitutes a contribution. It is a validity check on v3.